# 🌍 MODELO PREDICTIVO HÍBRIDO
### FÍSICA (FOURIER) + COMPORTAMIENTO (MACHINE LEARNING)
---
Este notebook une datos del ecosistema de Alicante (agua, clima, satélite NDVI y turismo) para predecir el consumo de agua mediante la combinación de estacionalidad matemática y variables exógenas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [ ]:
# 1. CARGA DE DATOS REALES (Agua, Clima y Satélite)
print("1. Uniendo datos del ecosistema de Alicante...")
df_agua = pd.read_csv('AMAEM-2022-2024.csv')
df_clima = pd.read_csv('clima_barrios_alicante_final.csv')
df_ndvi = pd.read_csv('ndvi_alicante.csv')

# Formatear fechas a Año-Mes
df_agua['fecha_mes'] = pd.to_datetime(df_agua['fecha']).dt.to_period('M').astype(str)
df_clima['fecha_mes'] = df_clima['fecha'].astype(str)

# Agrupar a nivel ciudad
df_ts = df_agua.groupby('fecha_mes')['consumo'].sum().reset_index()
df_clima_ciudad = df_clima.groupby('fecha_mes')[['tm_mes', 'p_mes']].mean().reset_index()

# Unir todas las fuentes de datos
df_final = pd.merge(df_ts, df_clima_ciudad, on='fecha_mes', how='inner')
df_final = pd.merge(df_final, df_ndvi, on='fecha_mes', how='inner')

# Extraemos el mes numérico
df_final['mes_num'] = pd.to_datetime(df_final['fecha_mes']).dt.month

# Para el turismo, usamos la simulación del aeropuerto
df_final['pasajeros_aeropuerto'] = df_final['mes_num'].apply(
    lambda x: 1600000 if x in [7, 8] else (1200000 if x in [6, 9] else 750000 + np.random.randint(-50000, 50000))
)

In [ ]:
# 2. MOTOR MATEMÁTICO (FÍSICA - FOURIER)
print("2. Calculando la estacionalidad base...")
t_historico = np.arange(len(df_final))
consumo_real = df_final['consumo'].values

def modelo_fourier(t, m, c, a1, b1, a2, b2):
    w = 2 * np.pi / 12
    return (m * t + c) + (a1 * np.cos(w * t) + b1 * np.sin(w * t)) + (a2 * np.cos(2 * w * t) + b2 * np.sin(2 * w * t))

coef_fourier, _ = curve_fit(modelo_fourier, t_historico, consumo_real, p0=[0, np.mean(consumo_real), 100000, 100000, 10000, 10000], maxfev=20000)
prediccion_fourier = modelo_fourier(t_historico, *coef_fourier)

In [ ]:
# 3. MACHINE LEARNING (PREDICIENDO EL COMPORTAMIENTO)
print("3. Entrenando IA con Clima y Satélite Copernicus...")
df_final['residuo'] = df_final['consumo'] - prediccion_fourier # Lo que la matemática no pilla

# La IA aprende de tus variables
X = df_final[['tm_mes', 'p_mes', 'pasajeros_aeropuerto', 'ndvi_satelite']]
y = df_final['residuo']

ml_model = RandomForestRegressor(n_estimators=100, random_state=42)
ml_model.fit(X, y)
impacto_exogeno = ml_model.predict(X)

# 4. RESULTADO HÍBRIDO FINAL
prediccion_hibrida = prediccion_fourier + impacto_exogeno

print("\n--- MÉTRICAS PARA EL JURADO ---")
print(f"Precisión de Fourier (Solo calendario): {r2_score(consumo_real, prediccion_fourier):.4f}")
print(f"Precisión Híbrida (IA + Clima + Satélite + Turismo): {r2_score(consumo_real, prediccion_hibrida):.4f}")

In [ ]:
# 5. GRÁFICA PARA LA PRESENTACIÓN
plt.figure(figsize=(14, 6))
plt.plot(df_final['fecha_mes'], consumo_real, 'ko-', label='Consumo Real de Alicante', linewidth=2)
plt.plot(df_final['fecha_mes'], prediccion_fourier, 'b--', alpha=0.5, label='Física (Solo Ecuación Matemática)')
plt.plot(df_final['fecha_mes'], prediccion_hibrida, 'r-', linewidth=3, label='Híbrido (Matemática + Machine Learning + Satélite)')

plt.title('Modelo de Predicción de Demanda de Agua: Mejora mediante Variables Exógenas', fontsize=16)
plt.xticks(rotation=45)
plt.ylabel('Consumo de Agua')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()